# Phase 7 - Missingness / Measurement Availability Feature Block

Experiment: `phase7_missingness_availability_v1`

This notebook executes only the ratified Phase 7 missingness / measurement
availability ladder. It does not generate submissions, run HPO, compare model
families, use public leaderboard feedback, use external data, use `School` as a
model feature, or modify the main experiment log.

## 1. Scope, Imports, and Configuration

**Purpose.** Pin the execution contract, paths, model, variants, thresholds, and
forbidden operations.

**Expected output.** Environment and policy constants ready for integrity gates.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import subprocess
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "data" / "input" / "train.csv").exists() and (candidate / ".git").exists():
            return candidate
    raise RuntimeError(f"Could not locate project root from {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
EXPERIMENT_ID = "phase7_missingness_availability_v1"
PROJECT_SEED = 42
RARE_SCHOOL_THRESHOLD = 5
F0_EXPECTED_OOF_AUC = 0.726616
F0_TOLERANCE = 1e-6
OOF_GAIN_THRESHOLD = 0.005436
SAME_SIGN_FOLD_THRESHOLD = 4
MIN_SLICE_SIZE = 50
SLICE_DEGRADATION_THRESHOLD = 0.02
F4_AUTHORIZED = False

DATA_DIR = PROJECT_ROOT / "data" / "input"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FOLDS_PATH = OUTPUTS_DIR / "folds" / "phase6_rf_sanity_baseline_v1_fold_assignments.csv"
PHASE6_OOF_PATH = OUTPUTS_DIR / "oof" / "phase6_rf_sanity_baseline_v1_oof_predictions.csv"
PHASE6A_V7_OOF_PATH = OUTPUTS_DIR / "oof" / "phase6a_v7_phase6_mean_impute_oof_predictions.csv"
MAIN_LOG_PATH = PROJECT_ROOT / "logs" / "experiment_log.csv"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

BASE_FEATURES = [
    "Year",
    "Age",
    "Height",
    "Weight",
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle",
    "Player_Type",
    "Position_Type",
    "Position",
]
CATEGORICAL_FEATURES = ["Player_Type", "Position_Type", "Position"]
PHYSICAL_TEST_COLUMNS = [
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle",
]
MISSINGNESS_FLAGS = [
    "Age_missing",
    "Sprint_40yd_missing",
    "Vertical_Jump_missing",
    "Bench_Press_Reps_missing",
    "Broad_Jump_missing",
    "Agility_3cone_missing",
    "Shuttle_missing",
]
EXPECTED_MISSING_COUNTS = {
    "Age": 435,
    "Sprint_40yd": 145,
    "Vertical_Jump": 554,
    "Bench_Press_Reps": 721,
    "Broad_Jump": 581,
    "Agility_3cone": 970,
    "Shuttle": 912,
}

VARIANT_ORDER = [
    "phase7_f0_anchor_recheck",
    "phase7_f1_median_flags",
    "phase7_f5_mean_flags",
    "phase7_f2_median_flags_count",
    "phase7_f3_median_flags_count_bins",
]
POSSIBLE_GATED_VARIANTS = ["phase7_f6_mean_flags_count"]

ARTIFACT_PATHS = {
    "variant_summary": OUTPUTS_DIR / "validation" / f"{EXPERIMENT_ID}_variant_summary.csv",
    "slice_report": OUTPUTS_DIR / "validation" / f"{EXPERIMENT_ID}_slice_report.csv",
    "validation_report": OUTPUTS_DIR / "reports" / f"{EXPERIMENT_ID}_validation_report.md",
    "candidate_log": OUTPUTS_DIR / "reports" / f"{EXPERIMENT_ID}_experiment_log_candidate.csv",
}
OOF_PATHS = {
    variant_id: OUTPUTS_DIR / "oof" / f"{EXPERIMENT_ID}_{variant_id}_oof_predictions.csv"
    for variant_id in VARIANT_ORDER + POSSIBLE_GATED_VARIANTS
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"numpy: {np.__version__}; pandas: {pd.__version__}; sklearn: {sklearn.__version__}")

Project root: C:\GitHub\reto_Tokio
Python: 3.13.13
numpy: 2.4.6; pandas: 3.0.3; sklearn: 1.9.0


## 2. Integrity Gates

**Purpose.** Fail before model execution if paths, git state, artifacts, or
frozen fold files are not exactly in the authorized state.

**Expected output.** A compact gate dictionary. Any failure raises an exception.

In [2]:
def run_git(args: list[str]) -> str:
    result = subprocess.run(["git", *args], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True)
    return result.stdout.strip()


def sha256_16(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()[:16]


def assert_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Required {label} missing: {path}")


def assert_not_exists(path: Path, label: str) -> None:
    if path.exists():
        raise FileExistsError(f"Artifact collision for {label}: {path}")


def read_bytes_if_exists(path: Path) -> bytes:
    return path.read_bytes() if path.exists() else b"__MISSING__"


head = run_git(["rev-parse", "--short", "HEAD"])
status_short = run_git(["status", "--short"])
diff_check = subprocess.run(["git", "diff", "--check"], cwd=PROJECT_ROOT, capture_output=True, text=True)
forbidden_diff = run_git([
    "diff",
    "--name-only",
    "--",
    "data/input",
    "notebooks/_official",
    "references",
    "outputs/submissions",
    "logs/experiment_log.csv",
    ".vscode/settings.json",
])

if head != "8b21db5":
    raise RuntimeError(f"Authorized HEAD mismatch: expected 8b21db5, got {head}")
if diff_check.returncode != 0 or diff_check.stdout.strip() or diff_check.stderr.strip():
    raise RuntimeError(f"git diff --check failed: {diff_check.stdout}{diff_check.stderr}")
if forbidden_diff:
    raise RuntimeError(f"Forbidden path diff before execution: {forbidden_diff}")
if any(line[:2] in {"M ", "A ", "D ", "R ", "C ", "U "} for line in status_short.splitlines()):
    raise RuntimeError(f"Unexpected staged changes before execution:\n{status_short}")

for path, label in [
    (TRAIN_PATH, "train.csv"),
    (TEST_PATH, "test.csv"),
    (SAMPLE_SUBMISSION_PATH, "sample_submission.csv"),
    (FOLDS_PATH, "frozen folds"),
    (PHASE6_OOF_PATH, "Phase 6 OOF reference"),
    (PHASE6A_V7_OOF_PATH, "Phase 6A V7 OOF reference"),
    (MAIN_LOG_PATH, "main experiment log"),
]:
    assert_exists(path, label)

for label, path in ARTIFACT_PATHS.items():
    assert_not_exists(path, label)
for variant_id, path in OOF_PATHS.items():
    assert_not_exists(path, f"OOF {variant_id}")

unauthorized_f4_paths = sorted((OUTPUTS_DIR / "oof").glob(f"{EXPERIMENT_ID}_phase7_f4*.csv"))
if unauthorized_f4_paths:
    raise RuntimeError(f"Unauthorized pre-existing F4 artifacts: {unauthorized_f4_paths}")

main_log_before = read_bytes_if_exists(MAIN_LOG_PATH)
gate_summary = {
    "head": head,
    "forbidden_diff_clean": forbidden_diff == "",
    "diff_check_clean": diff_check.returncode == 0,
    "status_untracked_only": all(line.startswith("?? ") for line in status_short.splitlines() if line),
}
gate_summary

{'head': '8b21db5',
 'forbidden_diff_clean': True,
 'diff_check_clean': True,
 'status_untracked_only': True}

## 3. Data Contract and Frozen Fold Integrity

**Purpose.** Load only official CSVs and frozen folds, confirm IDs, target,
columns, sample submission shape, and fold hash.

**Expected output.** Train/test/sample/folds loaded with hard integrity asserts.

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
folds = pd.read_csv(FOLDS_PATH)
phase6_oof_ref = pd.read_csv(PHASE6_OOF_PATH)
phase6a_v7_oof_ref = pd.read_csv(PHASE6A_V7_OOF_PATH)

if train.shape != (2781, 16):
    raise RuntimeError(f"Unexpected train shape: {train.shape}")
if test.shape != (696, 15):
    raise RuntimeError(f"Unexpected test shape: {test.shape}")
if sample_submission.shape != (696, 2):
    raise RuntimeError(f"Unexpected sample submission shape: {sample_submission.shape}")
if list(sample_submission.columns) != ["Id", "Drafted"]:
    raise RuntimeError(f"Unexpected sample submission columns: {list(sample_submission.columns)}")
if "Drafted" not in train.columns or "Drafted" in test.columns:
    raise RuntimeError("Target column contract failed.")
if not {"Id", "Drafted", "School"}.issubset(train.columns):
    raise RuntimeError("Required Id/Drafted/School columns missing from train.")
if not {"Id", "School"}.issubset(test.columns):
    raise RuntimeError("Required Id/School columns missing from test.")
if not sample_submission["Id"].equals(test["Id"]):
    raise RuntimeError("Sample submission Id order does not match test Id order.")
if sorted(train["Drafted"].dropna().unique().tolist()) != [0, 1]:
    raise RuntimeError("Train target is not binary with labels 0 and 1.")
if train.duplicated().sum() != 0 or test.duplicated().sum() != 0:
    raise RuntimeError("Duplicate rows found in official train/test.")

fold_hash = sha256_16(FOLDS_PATH)
if fold_hash != "96937649526bcadb":
    raise RuntimeError(f"Frozen fold hash mismatch: {fold_hash}")
if folds.shape != (2781, 2):
    raise RuntimeError(f"Unexpected fold file shape: {folds.shape}")
if list(folds.columns) != ["Id", "fold"]:
    raise RuntimeError(f"Unexpected fold file columns: {list(folds.columns)}")
if not folds["Id"].equals(train["Id"]):
    raise RuntimeError("Frozen fold Id order does not match train Id order.")
if sorted(folds["fold"].unique().tolist()) != [0, 1, 2, 3, 4]:
    raise RuntimeError("Frozen fold labels are not exactly 0..4.")

for ref_name, ref_df in [("phase6", phase6_oof_ref), ("phase6a_v7", phase6a_v7_oof_ref)]:
    required_cols = {"Id", "fold", "y_true", "y_pred_proba"}
    if not required_cols.issubset(ref_df.columns):
        raise RuntimeError(f"{ref_name} OOF missing columns: {required_cols - set(ref_df.columns)}")
    ref_sorted = ref_df.sort_values("Id").reset_index(drop=True)
    train_sorted_ids = train[["Id"]].sort_values("Id").reset_index(drop=True)
    if not ref_sorted["Id"].equals(train_sorted_ids["Id"]):
        raise RuntimeError(f"{ref_name} OOF Ids do not match train Ids.")

fold_array = folds["fold"].to_numpy()
y = train["Drafted"].astype(int).to_numpy()

data_contract_summary = {
    "train_shape": train.shape,
    "test_shape": test.shape,
    "sample_submission_shape": sample_submission.shape,
    "fold_hash": fold_hash,
    "fold_counts": folds["fold"].value_counts().sort_index().to_dict(),
}
data_contract_summary

{'train_shape': (2781, 16),
 'test_shape': (696, 15),
 'sample_submission_shape': (696, 2),
 'fold_hash': '96937649526bcadb',
 'fold_counts': {0: 557, 1: 556, 2: 556, 3: 556, 4: 556}}

## 4. Row-Wise Feature Builders and Sanity Checks

**Purpose.** Create only pre-registered, row-wise missingness and measurement
availability features. No fitting, target use, School encoding, or test-derived
statistics are allowed.

**Expected output.** Feature-augmented train copy and missingness assertions.

In [4]:
def add_rowwise_phase7_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Age_missing"] = out["Age"].isna().astype(int)
    for column in PHYSICAL_TEST_COLUMNS:
        out[f"{column}_missing"] = out[column].isna().astype(int)
    out["available_measurement_count"] = out[PHYSICAL_TEST_COLUMNS].notna().sum(axis=1).astype(int)
    count = out["available_measurement_count"]
    out["measurement_completeness_group"] = np.select(
        [count.eq(0), count.between(1, 3), count.between(4, 5), count.eq(6)],
        [0, 1, 2, 3],
        default=-1,
    ).astype(int)
    if (out["measurement_completeness_group"] < 0).any():
        raise RuntimeError("Invalid measurement_completeness_group value.")
    return out


for column in BASE_FEATURES:
    if column not in train.columns:
        raise RuntimeError(f"Base feature missing from train: {column}")
for column in PHYSICAL_TEST_COLUMNS + ["Age"]:
    observed = int(train[column].isna().sum())
    expected = EXPECTED_MISSING_COUNTS[column]
    if observed != expected:
        raise RuntimeError(f"Missing count mismatch for {column}: expected {expected}, got {observed}")

train_fe = add_rowwise_phase7_features(train)

flag_to_source = {
    "Age_missing": "Age",
    **{f"{column}_missing": column for column in PHYSICAL_TEST_COLUMNS},
}
for flag, source in flag_to_source.items():
    flag_sum = int(train_fe[flag].sum())
    expected = EXPECTED_MISSING_COUNTS[source]
    if flag_sum != expected:
        raise RuntimeError(f"Flag sum mismatch for {flag}: expected {expected}, got {flag_sum}")

if not train_fe["available_measurement_count"].between(0, 6).all():
    raise RuntimeError("available_measurement_count outside 0..6")
if not train_fe["measurement_completeness_group"].between(0, 3).all():
    raise RuntimeError("measurement_completeness_group outside 0..3")

school_key = train_fe["School"].astype("string").fillna("__MISSING__")
school_counts = school_key.value_counts(dropna=False)
train_fe["frequent_vs_rare_school_group"] = np.where(
    school_key.map(school_counts).to_numpy() < RARE_SCHOOL_THRESHOLD,
    "rare",
    "frequent",
)

feature_builder_summary = {
    "missing_flag_sums": {flag: int(train_fe[flag].sum()) for flag in MISSINGNESS_FLAGS},
    "available_measurement_count_counts": train_fe["available_measurement_count"].value_counts().sort_index().to_dict(),
    "measurement_completeness_group_counts": train_fe["measurement_completeness_group"].value_counts().sort_index().to_dict(),
    "school_group_counts": train_fe["frequent_vs_rare_school_group"].value_counts().to_dict(),
}
feature_builder_summary

{'missing_flag_sums': {'Age_missing': 435,
  'Sprint_40yd_missing': 145,
  'Vertical_Jump_missing': 554,
  'Bench_Press_Reps_missing': 721,
  'Broad_Jump_missing': 581,
  'Agility_3cone_missing': 970,
  'Shuttle_missing': 912},
 'available_measurement_count_counts': {0: 56,
  1: 240,
  2: 236,
  3: 137,
  4: 269,
  5: 454,
  6: 1389},
 'measurement_completeness_group_counts': {0: 56, 1: 613, 2: 723, 3: 1389},
 'school_group_counts': {'frequent': 2559, 'rare': 222}}

## 5. Fold-Safe Pipeline Helpers

**Purpose.** Define the audited fold-safe training loop, positive-class
probability extraction, metric calculation, and artifact validation.

**Expected output.** Helper functions; no model has been trained before the
variant ladder cell.

In [5]:
def get_git_dirty_label() -> str:
    short = run_git(["status", "--short"])
    return "dirty_untracked_only" if short else "clean"


def make_pipeline(feature_columns: list[str], impute_strategy: str) -> Pipeline:
    forbidden = {"Id", "Drafted", "School", "frequent_vs_rare_school_group"}
    overlap = sorted(forbidden.intersection(feature_columns))
    if overlap:
        raise RuntimeError(f"Forbidden columns in feature matrix: {overlap}")

    missing = [column for column in feature_columns if column not in train_fe.columns]
    if missing:
        raise RuntimeError(f"Feature columns missing from train_fe: {missing}")

    categorical_cols = [column for column in CATEGORICAL_FEATURES if column in feature_columns]
    numeric_cols = [column for column in feature_columns if column not in categorical_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", SimpleImputer(strategy=impute_strategy), numeric_cols),
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical_cols,
            ),
        ],
        remainder="drop",
    )
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=PROJECT_SEED,
        n_jobs=-1,
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("model", model)])


def positive_class_proba(fitted_pipeline: Pipeline, X_valid: pd.DataFrame) -> np.ndarray:
    model = fitted_pipeline.named_steps["model"]
    classes = np.asarray(model.classes_)
    matches = np.where(classes == 1)[0]
    if len(matches) != 1:
        raise RuntimeError(f"Expected class label 1 exactly once, got classes_={classes.tolist()}")
    proba = fitted_pipeline.predict_proba(X_valid)[:, int(matches[0])]
    if not np.isfinite(proba).all():
        raise RuntimeError("Predicted probabilities contain non-finite values.")
    if np.isnan(proba).any():
        raise RuntimeError("Predicted probabilities contain NaN.")
    if (proba < 0).any() or (proba > 1).any():
        raise RuntimeError("Predicted probabilities outside [0, 1].")
    return proba


def compute_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    if len(np.unique(y_true)) != 2:
        raise RuntimeError("ROC-AUC requested on one-class labels.")
    return float(roc_auc_score(y_true, y_score))


def validate_oof_df(oof_df: pd.DataFrame, variant_id: str) -> None:
    expected_cols = ["Id", "fold", "y_true", "y_pred_proba"]
    if list(oof_df.columns) != expected_cols:
        raise RuntimeError(f"{variant_id} OOF columns are not {expected_cols}: {list(oof_df.columns)}")
    if len(oof_df) != len(train):
        raise RuntimeError(f"{variant_id} OOF row count mismatch: {len(oof_df)}")
    if not oof_df["Id"].equals(train["Id"]):
        raise RuntimeError(f"{variant_id} OOF Id order does not match train order.")
    if not np.array_equal(oof_df["fold"].to_numpy(), fold_array):
        raise RuntimeError(f"{variant_id} OOF fold labels do not match frozen folds.")
    if not np.array_equal(oof_df["y_true"].to_numpy(), y):
        raise RuntimeError(f"{variant_id} OOF y_true does not match train target.")
    proba = oof_df["y_pred_proba"].to_numpy()
    if not np.isfinite(proba).all() or np.isnan(proba).any() or (proba < 0).any() or (proba > 1).any():
        raise RuntimeError(f"{variant_id} OOF probabilities invalid.")


def train_variant(variant_id: str, feature_columns: list[str], impute_strategy: str) -> dict:
    X = train_fe[feature_columns].copy()
    oof = np.full(len(train_fe), np.nan, dtype=float)
    fold_scores: list[float] = []
    fold_rows: list[dict] = []

    for fold_id in range(5):
        valid_mask = fold_array == fold_id
        train_mask = ~valid_mask
        y_train = y[train_mask]
        y_valid = y[valid_mask]
        if len(np.unique(y_train)) != 2 or len(np.unique(y_valid)) != 2:
            raise RuntimeError(f"{variant_id} fold {fold_id} has a one-class train or valid split.")

        pipeline = make_pipeline(feature_columns, impute_strategy)
        pipeline.fit(X.loc[train_mask], y_train)
        valid_proba = positive_class_proba(pipeline, X.loc[valid_mask])
        fold_auc = compute_auc(y_valid, valid_proba)
        oof[valid_mask] = valid_proba
        fold_scores.append(fold_auc)
        fold_rows.append(
            {
                "variant_id": variant_id,
                "fold": fold_id,
                "n_valid": int(valid_mask.sum()),
                "n_positive": int(y_valid.sum()),
                "n_negative": int(len(y_valid) - y_valid.sum()),
                "positive_rate": float(y_valid.mean()),
                "roc_auc": fold_auc,
            }
        )

    if np.isnan(oof).any():
        raise RuntimeError(f"{variant_id} has missing OOF predictions.")

    oof_df = pd.DataFrame(
        {
            "Id": train["Id"],
            "fold": fold_array,
            "y_true": y,
            "y_pred_proba": oof,
        }
    )
    validate_oof_df(oof_df, variant_id)
    oof_auc = compute_auc(y, oof)
    return {
        "variant_id": variant_id,
        "feature_columns": feature_columns,
        "impute_strategy": impute_strategy,
        "oof_df": oof_df,
        "oof_auc": oof_auc,
        "fold_scores": fold_scores,
        "fold_rows": fold_rows,
    }


def ref_auc(ref_df: pd.DataFrame) -> float:
    ref_ordered = ref_df.set_index("Id").loc[train["Id"]].reset_index()
    if not np.array_equal(ref_ordered["y_true"].to_numpy(), y):
        raise RuntimeError("Reference OOF y_true mismatch.")
    return compute_auc(y, ref_ordered["y_pred_proba"].to_numpy())


def paired_fold_deltas(result: dict, baseline_result: dict) -> list[float]:
    deltas = []
    for fold_id in range(5):
        valid_mask = fold_array == fold_id
        variant_auc = compute_auc(
            y[valid_mask],
            result["oof_df"].loc[valid_mask, "y_pred_proba"].to_numpy(),
        )
        baseline_auc = compute_auc(
            y[valid_mask],
            baseline_result["oof_df"].loc[valid_mask, "y_pred_proba"].to_numpy(),
        )
        deltas.append(float(variant_auc - baseline_auc))
    return deltas

## 6. Execute Pre-Registered Variant Ladder

**Purpose.** Run F0/F1/F5/F2/F3, then run F6 only if the ratified gate opens.
F4 is not authorized in this run.

**Expected output.** In-memory OOF predictions and fold metrics; no artifacts are
written until all gates have passed.

In [6]:
base_features = BASE_FEATURES.copy()
flags_features = base_features + MISSINGNESS_FLAGS
flags_count_features = flags_features + ["available_measurement_count"]
flags_count_bins_features = flags_count_features + ["measurement_completeness_group"]

variant_specs = [
    ("phase7_f0_anchor_recheck", base_features, "median"),
    ("phase7_f1_median_flags", flags_features, "median"),
    ("phase7_f5_mean_flags", flags_features, "mean"),
    ("phase7_f2_median_flags_count", flags_count_features, "median"),
    ("phase7_f3_median_flags_count_bins", flags_count_bins_features, "median"),
]

trained_results: dict[str, dict] = {}
execution_notes: list[str] = []

for variant_id, feature_columns, impute_strategy in variant_specs:
    print(f"Training {variant_id}...")
    trained_results[variant_id] = train_variant(variant_id, feature_columns, impute_strategy)

f0 = trained_results["phase7_f0_anchor_recheck"]
f0_oof_auc = f0["oof_auc"]
phase6_ref_auc = ref_auc(phase6_oof_ref)
phase6_v7_ref_auc = ref_auc(phase6a_v7_oof_ref)
phase6_ref_ordered = phase6_oof_ref.set_index("Id").loc[train["Id"]].reset_index()
f0_ref_max_abs_diff = float(np.max(np.abs(f0["oof_df"]["y_pred_proba"].to_numpy() - phase6_ref_ordered["y_pred_proba"].to_numpy())))

if abs(f0_oof_auc - F0_EXPECTED_OOF_AUC) > F0_TOLERANCE:
    raise RuntimeError(
        f"F0 anchor reproduction failed: expected {F0_EXPECTED_OOF_AUC} +/- {F0_TOLERANCE}, got {f0_oof_auc}"
    )
if abs(f0_oof_auc - phase6_ref_auc) > F0_TOLERANCE:
    raise RuntimeError(
        f"F0 does not match Phase 6 reference AUC within tolerance: F0={f0_oof_auc}, ref={phase6_ref_auc}"
    )

f5_minus_f1 = (
    trained_results["phase7_f5_mean_flags"]["oof_auc"]
    - trained_results["phase7_f1_median_flags"]["oof_auc"]
)
f6_run = False
if f5_minus_f1 >= OOF_GAIN_THRESHOLD:
    print("F6 gate opened; training phase7_f6_mean_flags_count.")
    trained_results["phase7_f6_mean_flags_count"] = train_variant(
        "phase7_f6_mean_flags_count",
        flags_count_features,
        "mean",
    )
    f6_run = True
else:
    execution_notes.append(
        f"F6 not run: F5-F1 OOF delta {f5_minus_f1:.6f} < threshold {OOF_GAIN_THRESHOLD:.6f}."
    )

if not F4_AUTHORIZED:
    execution_notes.append("F4 not run: role interactions were not authorized in the Project Authorization Note.")

{
    "f0_oof_auc": f0_oof_auc,
    "phase6_ref_auc": phase6_ref_auc,
    "phase6_v7_ref_auc": phase6_v7_ref_auc,
    "f0_vs_phase6_ref_max_abs_proba_diff": f0_ref_max_abs_diff,
    "f5_minus_f1": f5_minus_f1,
    "f6_run": f6_run,
}

Training phase7_f0_anchor_recheck...


Training phase7_f1_median_flags...


Training phase7_f5_mean_flags...


Training phase7_f2_median_flags_count...


Training phase7_f3_median_flags_count_bins...


{'f0_oof_auc': 0.7266161714116555,
 'phase6_ref_auc': 0.7266161714116555,
 'phase6_v7_ref_auc': 0.802270868706666,
 'f0_vs_phase6_ref_max_abs_proba_diff': 5.551115123125783e-16,
 'f5_minus_f1': 0.0014982981102841242,
 'f6_run': False}

## 7. Slice Diagnostics and Acceptance Classification

**Purpose.** Apply the pre-registered OOF gain, same-sign fold, and mandatory
slice-guard criteria.

**Expected output.** Variant summary and slice report tables.

In [7]:
SLICE_COLUMNS = [
    "Player_Type",
    "Position_Type",
    "Year",
    "Age_missing",
    "available_measurement_count",
    "measurement_completeness_group",
    "frequent_vs_rare_school_group",
]


def sorted_slice_values(series: pd.Series) -> list:
    values = series.dropna().unique().tolist()
    try:
        return sorted(values)
    except TypeError:
        return sorted(values, key=lambda value: str(value))


def build_slice_rows(variant_id: str, oof_df: pd.DataFrame) -> list[dict]:
    rows = []
    for slice_name in SLICE_COLUMNS:
        if slice_name not in train_fe.columns:
            rows.append(
                {
                    "variant_id": variant_id,
                    "slice_name": slice_name,
                    "slice_value": "not_applicable",
                    "n": 0,
                    "n_positive": 0,
                    "n_negative": 0,
                    "positive_rate": np.nan,
                    "roc_auc": np.nan,
                    "status": "not_applicable",
                    "reason_if_skipped": "slice column missing",
                    "delta_vs_f0": np.nan,
                    "notes": "",
                }
            )
            continue
        for value in sorted_slice_values(train_fe[slice_name]):
            mask = train_fe[slice_name].eq(value).to_numpy()
            n = int(mask.sum())
            y_slice = y[mask]
            n_positive = int(y_slice.sum())
            n_negative = int(n - n_positive)
            positive_rate = float(y_slice.mean()) if n else np.nan
            status = "computed"
            reason = ""
            roc_auc = np.nan
            if n < MIN_SLICE_SIZE:
                status = "skipped_too_small"
                reason = f"n<{MIN_SLICE_SIZE}"
            elif len(np.unique(y_slice)) < 2:
                status = "skipped_one_class"
                reason = "one target class"
            else:
                roc_auc = compute_auc(y_slice, oof_df.loc[mask, "y_pred_proba"].to_numpy())
            rows.append(
                {
                    "variant_id": variant_id,
                    "slice_name": slice_name,
                    "slice_value": str(value),
                    "n": n,
                    "n_positive": n_positive,
                    "n_negative": n_negative,
                    "positive_rate": positive_rate,
                    "roc_auc": roc_auc,
                    "status": status,
                    "reason_if_skipped": reason,
                    "delta_vs_f0": np.nan,
                    "notes": "School grouping is diagnostic-only" if slice_name == "frequent_vs_rare_school_group" else "",
                }
            )
    return rows


slice_rows = []
for variant_id, result in trained_results.items():
    slice_rows.extend(build_slice_rows(variant_id, result["oof_df"]))
slice_report = pd.DataFrame(slice_rows)

f0_slice_lookup = (
    slice_report[
        (slice_report["variant_id"] == "phase7_f0_anchor_recheck")
        & (slice_report["status"] == "computed")
    ]
    .set_index(["slice_name", "slice_value"])["roc_auc"]
    .to_dict()
)

for idx, row in slice_report.iterrows():
    if row["variant_id"] == "phase7_f0_anchor_recheck" or row["status"] != "computed":
        continue
    key = (row["slice_name"], row["slice_value"])
    if key in f0_slice_lookup:
        slice_report.at[idx, "delta_vs_f0"] = float(row["roc_auc"] - f0_slice_lookup[key])

slice_guard_by_variant = {}
for variant_id in trained_results:
    if variant_id == "phase7_f0_anchor_recheck":
        slice_guard_by_variant[variant_id] = False
        continue
    variant_slices = slice_report[
        (slice_report["variant_id"] == variant_id)
        & (slice_report["status"] == "computed")
        & (slice_report["n"] >= MIN_SLICE_SIZE)
    ]
    degraded = variant_slices["delta_vs_f0"].dropna() < -SLICE_DEGRADATION_THRESHOLD
    slice_guard_by_variant[variant_id] = bool(degraded.any())

variant_summary_rows = []
fold_detail_rows = []
f0_auc = trained_results["phase7_f0_anchor_recheck"]["oof_auc"]
v7_gap = phase6_v7_ref_auc - f0_auc

for variant_id, result in trained_results.items():
    fold_scores = result["fold_scores"]
    fold_detail_rows.extend(result["fold_rows"])
    fold_deltas = [0.0] * 5 if variant_id == "phase7_f0_anchor_recheck" else paired_fold_deltas(result, f0)
    gain = float(result["oof_auc"] - f0_auc)
    same_sign = int(sum(delta > 0 for delta in fold_deltas))
    if variant_id == "phase7_f0_anchor_recheck":
        classification = "anchor_recheck"
        decision_reason = "F0 reproduced the Phase 6 anchor within tolerance."
    elif slice_guard_by_variant[variant_id]:
        classification = "escalated"
        decision_reason = f"At least one mandatory slice degraded by more than {SLICE_DEGRADATION_THRESHOLD:.2f} AUC."
    elif gain >= OOF_GAIN_THRESHOLD and same_sign >= SAME_SIGN_FOLD_THRESHOLD:
        classification = "adopted"
        decision_reason = "OOF gain and same-sign fold criteria passed, with slice guard clear."
    else:
        classification = "rejected"
        decision_reason = "Pre-registered OOF gain and/or same-sign fold criteria did not pass."
    recovery_vs_v7 = float(gain / v7_gap) if v7_gap != 0 else np.nan
    variant_summary_rows.append(
        {
            "experiment_id": EXPERIMENT_ID,
            "variant_id": variant_id,
            "run_status": "trained",
            "classification": classification,
            "decision_reason": decision_reason,
            "feature_count": len(result["feature_columns"]),
            "features": "|".join(result["feature_columns"]),
            "impute_strategy": result["impute_strategy"],
            "oof_auc": result["oof_auc"],
            "delta_vs_f0_oof": gain,
            "fold_mean_auc": float(np.mean(fold_scores)),
            "fold_std_auc": float(np.std(fold_scores, ddof=1)),
            "fold_auc_scores": "|".join(f"{score:.12f}" for score in fold_scores),
            "fold_deltas_vs_f0": "|".join(f"{delta:.12f}" for delta in fold_deltas),
            "same_sign_positive_folds": same_sign,
            "slice_guard_triggered": slice_guard_by_variant[variant_id],
            "recovery_fraction_vs_phase6a_v7": recovery_vs_v7,
        }
    )

if not f6_run:
    variant_summary_rows.append(
        {
            "experiment_id": EXPERIMENT_ID,
            "variant_id": "phase7_f6_mean_flags_count",
            "run_status": "not_run_gated",
            "classification": "not_run",
            "decision_reason": f"F6 gate did not open: F5-F1 OOF delta {f5_minus_f1:.12f} < {OOF_GAIN_THRESHOLD:.12f}.",
            "feature_count": len(flags_count_features),
            "features": "|".join(flags_count_features),
            "impute_strategy": "mean",
            "oof_auc": np.nan,
            "delta_vs_f0_oof": np.nan,
            "fold_mean_auc": np.nan,
            "fold_std_auc": np.nan,
            "fold_auc_scores": "",
            "fold_deltas_vs_f0": "",
            "same_sign_positive_folds": 0,
            "slice_guard_triggered": False,
            "recovery_fraction_vs_phase6a_v7": np.nan,
        }
    )

variant_summary = pd.DataFrame(variant_summary_rows)
fold_detail = pd.DataFrame(fold_detail_rows)

variant_summary[
    [
        "variant_id",
        "run_status",
        "classification",
        "oof_auc",
        "delta_vs_f0_oof",
        "fold_mean_auc",
        "fold_std_auc",
        "same_sign_positive_folds",
        "slice_guard_triggered",
    ]
]

,variant_id,run_status,classification,oof_auc,delta_vs_f0_oof,fold_mean_auc,fold_std_auc,same_sign_positive_folds,slice_guard_triggered
0,phase7_f0_anchor_recheck,trained,anchor_recheck,0.726616,0.000000,0.729253,0.030629,0,False
1,phase7_f1_median_flags,trained,escalated,0.811568,0.084952,0.813093,0.034002,5,True
2,phase7_f5_mean_flags,trained,escalated,0.813066,0.086450,0.813652,0.033866,5,True
3,phase7_f2_median_flags_count,trained,adopted,0.811650,0.085034,0.812456,0.029238,5,False
4,phase7_f3_median_flags_count_bins,trained,escalated,0.810606,0.083989,0.812605,0.028212,5,True
5,phase7_f6_mean_flags_count,not_run_gated,not_run,NaN,NaN,NaN,NaN,0,False


## 8. Artifact Writes and Final Report

**Purpose.** Persist only authorized Phase 7 artifacts after all gates have
passed. The main experiment log is verified unchanged.

**Expected output.** OOF files, summary CSV, slice report CSV, markdown report,
and candidate log CSV.

In [8]:
def fmt_float(value: float | int | np.floating | None, digits: int = 6) -> str:
    if value is None:
        return "Not confirmed yet"
    try:
        if pd.isna(value):
            return "Not confirmed yet"
    except TypeError:
        pass
    return f"{float(value):.{digits}f}"


def markdown_table(df: pd.DataFrame, columns: list[str]) -> str:
    view = df[columns].copy()
    for col in view.columns:
        if pd.api.types.is_float_dtype(view[col]):
            view[col] = view[col].map(lambda x: "" if pd.isna(x) else f"{x:.6f}")
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = ["| " + " | ".join(str(value) for value in row) + " |" for row in view.to_numpy()]
    return "\n".join([header, sep, *rows])


def top_slice_findings() -> pd.DataFrame:
    computed = slice_report[
        (slice_report["variant_id"] != "phase7_f0_anchor_recheck")
        & (slice_report["status"] == "computed")
        & (slice_report["delta_vs_f0"].notna())
    ].copy()
    if computed.empty:
        return computed
    computed["abs_delta"] = computed["delta_vs_f0"].abs()
    return computed.sort_values(["abs_delta", "variant_id"], ascending=[False, True]).head(15)


def write_csv_fail_closed(df: pd.DataFrame, path: Path, label: str) -> None:
    assert_not_exists(path, label)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


for variant_id, result in trained_results.items():
    write_csv_fail_closed(result["oof_df"], OOF_PATHS[variant_id], f"OOF {variant_id}")

write_csv_fail_closed(variant_summary, ARTIFACT_PATHS["variant_summary"], "variant summary")
write_csv_fail_closed(slice_report, ARTIFACT_PATHS["slice_report"], "slice report")

candidate_log_columns = [
    "experiment_id",
    "date",
    "phase",
    "notebook_or_script",
    "git_commit_or_status",
    "data_version",
    "fold_strategy",
    "random_seed",
    "feature_block",
    "preprocessing_summary",
    "model_family",
    "model_params_summary",
    "hpo_status",
    "cv_auc_mean",
    "cv_auc_std",
    "fold_scores",
    "slice_metrics_available",
    "leakage_checks_passed",
    "submission_created",
    "submission_path",
    "public_lb_score_if_submitted",
    "notes",
    "decision",
]
best_rows = variant_summary[variant_summary["classification"].eq("adopted")]
adopted_variants = ",".join(best_rows["variant_id"].tolist()) if not best_rows.empty else "none"
candidate_log = pd.DataFrame(
    [
        {
            "experiment_id": EXPERIMENT_ID,
            "date": date.today().isoformat(),
            "phase": "Phase 7 - Missingness / Measurement Availability Feature Block",
            "notebook_or_script": "notebooks/07_phase7_missingness_availability_feature_block.ipynb",
            "git_commit_or_status": f"{head}; {get_git_dirty_label()}",
            "data_version": "official_data_input_train_test_sample_submission",
            "fold_strategy": "frozen_phase6_stratified_kfold_5_seed42",
            "random_seed": PROJECT_SEED,
            "feature_block": "missingness_measurement_availability",
            "preprocessing_summary": "Fold-safe ColumnTransformer; numeric SimpleImputer median/mean by variant; role categorical most_frequent+OHE; School excluded.",
            "model_family": "RandomForestClassifier",
            "model_params_summary": "n_estimators=100; max_depth=5; random_state=42; n_jobs=-1",
            "hpo_status": "not_run",
            "cv_auc_mean": variant_summary.loc[variant_summary["variant_id"].eq("phase7_f0_anchor_recheck"), "fold_mean_auc"].iloc[0],
            "cv_auc_std": variant_summary.loc[variant_summary["variant_id"].eq("phase7_f0_anchor_recheck"), "fold_std_auc"].iloc[0],
            "fold_scores": variant_summary.loc[variant_summary["variant_id"].eq("phase7_f0_anchor_recheck"), "fold_auc_scores"].iloc[0],
            "slice_metrics_available": True,
            "leakage_checks_passed": not bool(variant_summary["classification"].eq("escalated").any()),
            "submission_created": False,
            "submission_path": "",
            "public_lb_score_if_submitted": "",
            "notes": "; ".join(execution_notes + [f"adopted_variants={adopted_variants}"]),
            "decision": f"adopted_variants={adopted_variants}; acceptance remains user-gated",
        }
    ],
    columns=candidate_log_columns,
)
write_csv_fail_closed(candidate_log, ARTIFACT_PATHS["candidate_log"], "candidate log")

slice_findings = top_slice_findings()
classification_counts = variant_summary["classification"].value_counts(dropna=False).to_dict()
escalated_variants = variant_summary.loc[variant_summary["classification"].eq("escalated"), "variant_id"].tolist()

report_lines = [
    f"# Phase 7 Validation Report - {EXPERIMENT_ID}",
    "",
    "## Scope",
    "",
    "This run executed only the pre-registered missingness / measurement availability feature block. It did not generate submissions, run HPO, compare model families, use public leaderboard feedback, use external data, use School as a model feature, modify `logs/experiment_log.csv`, or open Phase 8.",
    "",
    "## Environment and Integrity Gates",
    "",
    f"- Git HEAD: `{head}`",
    f"- Git status label: `{get_git_dirty_label()}`",
    f"- Python: `{sys.version.split()[0]}`",
    f"- numpy: `{np.__version__}`",
    f"- pandas: `{pd.__version__}`",
    f"- scikit-learn: `{sklearn.__version__}`",
    f"- Frozen fold sha256[:16]: `{fold_hash}`",
    f"- Main experiment log unchanged: `pending final assertion`",
    f"- F0 reference max absolute probability delta vs Phase 6 OOF: `{f0_ref_max_abs_diff:.12f}`",
    "",
    "## Variant Summary",
    "",
    markdown_table(
        variant_summary,
        [
            "variant_id",
            "run_status",
            "classification",
            "oof_auc",
            "delta_vs_f0_oof",
            "fold_mean_auc",
            "fold_std_auc",
            "same_sign_positive_folds",
            "slice_guard_triggered",
        ],
    ),
    "",
    "## Fold Scores",
    "",
    markdown_table(fold_detail, ["variant_id", "fold", "n_valid", "n_positive", "n_negative", "positive_rate", "roc_auc"]),
    "",
    "## F1 vs F5 vs V7 Reference",
    "",
    f"- F0 OOF ROC-AUC: `{f0_oof_auc:.6f}`.",
    f"- Phase 6A V7 reference OOF ROC-AUC: `{phase6_v7_ref_auc:.6f}`.",
    f"- F1 OOF ROC-AUC: `{trained_results['phase7_f1_median_flags']['oof_auc']:.6f}`.",
    f"- F5 OOF ROC-AUC: `{trained_results['phase7_f5_mean_flags']['oof_auc']:.6f}`.",
    f"- F5 - F1 OOF delta: `{f5_minus_f1:.6f}`.",
    f"- F6 run: `{f6_run}`.",
    "The V7 reference is used only as historical context for how much of the mean-imputation signal is recovered by explicit row-wise availability features.",
    "",
    "## Slice Findings",
    "",
    markdown_table(
        slice_findings if not slice_findings.empty else pd.DataFrame(columns=["variant_id", "slice_name", "slice_value", "n", "roc_auc", "delta_vs_f0", "status"]),
        ["variant_id", "slice_name", "slice_value", "n", "roc_auc", "delta_vs_f0", "status"],
    ),
    "",
    "## Leakage Checklist",
    "",
    "- All preprocessing was fitted inside each training fold through `Pipeline` / `ColumnTransformer`.",
    "- Test data was used only for contract checks; no test fitting, tuning, selection, final inference, or submission occurred.",
    "- `School` was excluded from every model feature matrix and used only for the diagnostic frequent-vs-rare slice.",
    "- No target encoding, feature selection, dimensionality reduction, HPO, ensembles, model-family comparison, or leaderboard feedback was used.",
    "- Positive-class probabilities were extracted only after verifying `estimator.classes_` contained label `1` exactly once.",
    "- Report language is associative and predictive only, not causal.",
    "",
    "## Warnings and Decision State",
    "",
    f"- Classification counts: `{classification_counts}`.",
    f"- Escalated variants: `{escalated_variants if escalated_variants else 'none'}`.",
    "- F4 was not authorized and was not executed. Any F4 activation requires a new Project Authorization Note and run_id.",
    "- Acceptance and commit remain gated on explicit user/project director authorization.",
    "- Phase 8 remains locked.",
]
report_text = "\n".join(report_lines) + "\n"
assert_not_exists(ARTIFACT_PATHS["validation_report"], "validation report")
ARTIFACT_PATHS["validation_report"].parent.mkdir(parents=True, exist_ok=True)
ARTIFACT_PATHS["validation_report"].write_text(report_text.replace("pending final assertion", "true"), encoding="utf-8")

main_log_after = read_bytes_if_exists(MAIN_LOG_PATH)
if main_log_after != main_log_before:
    raise RuntimeError("logs/experiment_log.csv changed during Phase 7 execution.")

artifact_summary = {
    "oof_files_written": [str(OOF_PATHS[variant_id].relative_to(PROJECT_ROOT)) for variant_id in trained_results],
    "summary_path": str(ARTIFACT_PATHS["variant_summary"].relative_to(PROJECT_ROOT)),
    "slice_report_path": str(ARTIFACT_PATHS["slice_report"].relative_to(PROJECT_ROOT)),
    "validation_report_path": str(ARTIFACT_PATHS["validation_report"].relative_to(PROJECT_ROOT)),
    "candidate_log_path": str(ARTIFACT_PATHS["candidate_log"].relative_to(PROJECT_ROOT)),
    "main_log_unchanged": True,
}
artifact_summary

{'oof_files_written': ['outputs\\oof\\phase7_missingness_availability_v1_phase7_f0_anchor_recheck_oof_predictions.csv',
  'outputs\\oof\\phase7_missingness_availability_v1_phase7_f1_median_flags_oof_predictions.csv',
  'outputs\\oof\\phase7_missingness_availability_v1_phase7_f5_mean_flags_oof_predictions.csv',
  'outputs\\oof\\phase7_missingness_availability_v1_phase7_f2_median_flags_count_oof_predictions.csv',
  'outputs\\oof\\phase7_missingness_availability_v1_phase7_f3_median_flags_count_bins_oof_predictions.csv'],
 'summary_path': 'outputs\\validation\\phase7_missingness_availability_v1_variant_summary.csv',
 'slice_report_path': 'outputs\\validation\\phase7_missingness_availability_v1_slice_report.csv',
 'validation_report_path': 'outputs\\reports\\phase7_missingness_availability_v1_validation_report.md',
 'candidate_log_path': 'outputs\\reports\\phase7_missingness_availability_v1_experiment_log_candidate.csv',
 'main_log_unchanged': True}

## 9. Final Notebook Summary

Phase 7 execution is complete inside this notebook when all cells above pass.
Acceptance, staging, commit, and Phase 8 remain locked behind explicit
user/project director authorization.